# Complete Analysis Pipeline

This notebook demonstrates a complete protein motif analysis pipeline:

1. **UniProt API** - Retrieving protein sequences
2. **Sequence Motif Search** - Finding consensus patterns
3. **Structure Motif Search** - Finding 3D structural motifs
4. **Results Integration** - Combining results

---
## Setup

In [ ]:
import sys
import os
import pandas as pd
import json

sys.path.insert(0, os.path.abspath('../sequence_motif'))
sys.path.insert(0, os.path.abspath('../structure_motif'))

from file_converter import process_protein_files
from motif_searcher import run_motif_search
from uniprot_api import search_uniprot, to_csv
from search_3d_motif import search_single_file, parse_motif_file

PROTEIN_FILES_DIR = '../protein_files'
MOTIF_LIBRARIES_DIR = '../sequence_motif/motif_libraries'
STRUCTURE_MOTIFS_DIR = '../structure_motif/motifs'
OUTPUT_DIR = '../outputs/pipeline_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Setup complete!')

---
## Step 1: Retrieve Sequences from UniProt

In [ ]:
# Query UniProt for human serine proteases
query = '(organism_name:"Homo sapiens") AND (protein_name:"serine protease")'
print(f'Querying UniProt: {query}')
data = search_uniprot(query, limit=20)
uniprot_csv = os.path.join(OUTPUT_DIR, 'serine_proteases.csv')
to_csv(data, uniprot_csv)
print(f'Saved sequences')

In [ ]:
# View results
uniprot_df = pd.read_csv(uniprot_csv)
print(f'Retrieved {len(uniprot_df)} sequences')
uniprot_df[['Entry', 'Protein names']].head(10)

---
## Step 2: Convert Structure Files to Sequences

In [ ]:
# Convert protein files to sequences
structures_csv = os.path.join(OUTPUT_DIR, 'structure_sequences.csv')
process_protein_files(PROTEIN_FILES_DIR, structures_csv)
struct_df = pd.read_csv(structures_csv)
print(f'Converted {len(struct_df)} sequences')

---
## Step 3: Sequence Motif Search

In [ ]:
# Combine sequences from all sources
uniprot_seqs = uniprot_df[['Entry', 'sequence']].copy()
uniprot_seqs.columns = ['name', 'sequence']
struct_df_copy = struct_df[['name', 'sequence']].copy()
combined_df = pd.concat([uniprot_seqs, struct_df_copy], ignore_index=True)
combined_csv = os.path.join(OUTPUT_DIR, 'combined_sequences.csv')
combined_df.to_csv(combined_csv, index=False)
print(f'Combined {len(combined_df)} sequences')

In [ ]:
# Create custom motifs
motifs_df = pd.DataFrame({
    'motif_name': ['PKA_Site', 'PKC_Site', 'CK2_Site', 'Trypsin_Cleavage', 'RGD_Adhesion', 'NLS_Signal'],
    'motifs': ['RRx[ST]', '[ST]x[RK]', '[ST]xx[DE]', '[KR]', 'RGD', '[+][+]xx[+]']
})
motifs_file = os.path.join(OUTPUT_DIR, 'analysis_motifs.csv')
motifs_df.to_csv(motifs_file, index=False)
motifs_df

In [ ]:
# Run sequence motif search
original_dir = os.getcwd()
os.chdir('../sequence_motif')
run_motif_search(
    motifs_file=motifs_file,
    motif_column='motifs',
    motif_name_column='motif_name',
    sequences_file=combined_csv,
    sequence_column='sequence',
    output_file=os.path.join(OUTPUT_DIR, 'seq_results.csv'),
    name_column='name'
)
os.chdir(original_dir)
print('Sequence search complete!')

---
## Step 4: Structure Motif Search

In [ ]:
# Load catalytic triad motif definition
catalytic_motif = os.path.join(STRUCTURE_MOTIFS_DIR, 'catalytic_triad.json')
with open(catalytic_motif, 'r') as f:
    motif_def = json.load(f)
print(f"Motif: {motif_def['motif_name']}")
print(f"Components: {len(motif_def['components'])}")
print(f"Constraints: {len(motif_def['constraints'])}")

In [ ]:
# Search structures for catalytic triads
struct_results_dir = os.path.join(OUTPUT_DIR, 'structure_results')
os.makedirs(struct_results_dir, exist_ok=True)

structure_files = [os.path.join(PROTEIN_FILES_DIR, f) for f in os.listdir(PROTEIN_FILES_DIR) if f.endswith(('.pdb', '.cif'))]
print(f'Searching {len(structure_files)} structures...')

results = []
for sf in structure_files:
    found = search_single_file(sf, motif_def)
    result = {'file': os.path.basename(sf), 'count': len(found), 'matches': found}
    results.append(result)
    with open(os.path.join(struct_results_dir, os.path.basename(sf).rsplit('.', 1)[0] + '.json'), 'w') as f:
        json.dump(result, f, indent=2)
    if found:
        print(f"  {os.path.basename(sf)}: {len(found)} motifs found")

In [ ]:
# Summary
print('\n' + '='*50)
print('PIPELINE SUMMARY')
print('='*50)
print(f'UniProt sequences: {len(uniprot_df)}')
print(f'Structure sequences: {len(struct_df)}')
print(f'Structures with catalytic triads: {sum(1 for r in results if r["count"] > 0)}')
print(f'Total catalytic triads found: {sum(r["count"] for r in results)}')
print('='*50)

---
## Conclusion

This pipeline demonstrated:
- UniProt API integration
- File conversion from PDB/CIF
- Sequence motif searching with rich nomenclature
- 3D structural motif searching
- Result integration and summary